
# Temporally Regularized Multi-Image Richardson–Lucy Deconvolution

This notebook implements a **temporally regularized multi-image Richardson–Lucy (TR-MIRL)** method for a stack of repeated fluorescence images.

The method assumes that repeated frames contain the same underlying object, while frame-to-frame noise is largely independent. The additional regularization uses **only temporal statistics at each pixel across the stack of RL correction maps**.


For frame \(l\),

\[
y_l \sim \mathrm{Poisson}(H_l x + b_l),
\]

where:

- \(x\): latent fluorescence image,
- \(H_l\): PSF/convolution operator,
- \(b_l\): background for frame \(l\),
- \(y_l\): observed frame.

For every RL iteration, the frame-specific correction is

\[
C_l^{(k)}
=
\frac{
H_l^T
\left[
\dfrac{y_l}{H_l x^{(k)} + b_l + \epsilon}
\right]
}{
H_l^T \mathbf{1} + \epsilon
}.
\]

Standard multi-image RL uses

\[
\bar C^{(k)} = \frac{1}{L}\sum_l C_l^{(k)}.
\]

The temporally regularized method computes, independently at every pixel,

\[
\mu_C = \mathrm{mean}_l(C_l),
\qquad
\sigma_C^2 = \mathrm{var}_l(C_l),
\]

and defines a temporal confidence

\[
W =
\frac{\mu_C^2}
{\mu_C^2 + \alpha\sigma_C^2 + \epsilon}.
\]

The update is

\[
x^{(k+1)}
=
x^{(k)}
\left[
1 + W(\mu_C-1)
\right].
\]

Thus, a reproducible RL correction across time is applied strongly, while a temporally unstable correction is pushed toward an update factor of 1.


In [ ]:

# Install only if needed:
# %pip install numpy scipy matplotlib tifffile

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
import tifffile as tiff
from pathlib import Path

EPS = 1e-12


## 1. Helper functions

In [ ]:

def normalize_psf(psf):
    psf = np.asarray(psf, dtype=np.float64)
    psf = np.clip(psf, 0, None)
    s = psf.sum()
    if s <= 0:
        raise ValueError("PSF sum must be > 0.")
    return psf / s


def gaussian_psf(size=21, sigma=2.0):
    """Generate a normalized 2D Gaussian PSF."""
    if size % 2 == 0:
        size += 1
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return normalize_psf(psf)


def conv2(image, kernel):
    return fftconvolve(image, kernel, mode="same")


def adjoint_conv2(image, psf):
    """Adjoint convolution using the 180-degree rotated PSF."""
    return fftconvolve(image, psf[::-1, ::-1], mode="same")


def load_tiff_stack(path):
    """
    Load a multi-page TIFF as an array with shape (frames, y, x).
    """
    arr = tiff.imread(path).astype(np.float64)
    if arr.ndim == 2:
        arr = arr[None, ...]
    if arr.ndim != 3:
        raise ValueError(f"Expected a 2D image or 3D TIFF stack, got shape {arr.shape}.")
    return arr


def load_tiff_folder(folder, pattern="*.tif"):
    """
    Load individual TIFF frames from a folder.
    All images must have the same shape.
    """
    files = sorted(Path(folder).glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {folder}")
    imgs = [tiff.imread(f).astype(np.float64) for f in files]
    shapes = {im.shape for im in imgs}
    if len(shapes) != 1:
        raise ValueError("All frames must have the same shape.")
    return np.stack(imgs, axis=0), files


def estimate_scalar_backgrounds(stack, percentile=5):
    """
    Simple per-frame scalar background estimate.
    For quantitative work, a manually selected background ROI is preferable.
    """
    stack = np.asarray(stack, dtype=np.float64)
    return np.percentile(stack, percentile, axis=(1, 2))


def backgrounds_from_roi(stack, mask):
    """
    Estimate one scalar background value per frame from a Boolean background ROI.
    mask shape must be (y, x).
    """
    stack = np.asarray(stack, dtype=np.float64)
    mask = np.asarray(mask, dtype=bool)
    if mask.shape != stack.shape[1:]:
        raise ValueError("ROI mask must match the image dimensions.")
    return np.array([frame[mask].mean() for frame in stack], dtype=np.float64)



## 2. Core RL algorithms

The functions below implement:

1. **Single-image RL**
2. **Standard multi-image RL**
3. **Temporally regularized multi-image RL**

The temporal regularizer is calculated only along the frame axis of the RL correction stack.


In [ ]:

def _prepare_psfs(psf, n_frames):
    """Accept one common PSF or one PSF per frame."""
    if isinstance(psf, (list, tuple)):
        if len(psf) != n_frames:
            raise ValueError("Number of PSFs must equal number of frames.")
        return [normalize_psf(p) for p in psf]

    psf = np.asarray(psf)
    if psf.ndim == 2:
        p = normalize_psf(psf)
        return [p] * n_frames
    elif psf.ndim == 3 and psf.shape[0] == n_frames:
        return [normalize_psf(p) for p in psf]

    raise ValueError("psf must be 2D, a list of 2D PSFs, or shape (frames, py, px).")


def _prepare_backgrounds(backgrounds, stack):
    n_frames = stack.shape[0]

    if backgrounds is None:
        return np.zeros(n_frames, dtype=np.float64)

    b = np.asarray(backgrounds, dtype=np.float64)

    if b.ndim == 0:
        return np.full(n_frames, float(b))

    if b.ndim == 1 and len(b) == n_frames:
        return b

    if b.ndim == 3 and b.shape == stack.shape:
        return b

    raise ValueError(
        "backgrounds must be None, a scalar, one scalar per frame, "
        "or a background stack with the same shape as the image stack."
    )


def _background_for_frame(backgrounds, l):
    if np.asarray(backgrounds).ndim == 1:
        return backgrounds[l]
    return backgrounds[l]


def single_image_rl(image, psf, iterations=20, background=0.0, init=None, eps=EPS):
    image = np.asarray(image, dtype=np.float64)
    image = np.clip(image, 0, None)
    psf = normalize_psf(psf)

    if init is None:
        x = np.full_like(image, max(image.mean() - float(np.mean(background)), eps))
    else:
        x = np.clip(np.asarray(init, dtype=np.float64), eps, None)

    sensitivity = adjoint_conv2(np.ones_like(image), psf)

    for _ in range(iterations):
        pred = conv2(x, psf) + background
        ratio = image / np.maximum(pred, eps)
        correction = adjoint_conv2(ratio, psf)
        correction /= np.maximum(sensitivity, eps)
        x *= correction
        x = np.clip(x, 0, None)

    return x


def multi_image_rl(stack, psf, iterations=20, backgrounds=None, init=None, eps=EPS):
    """
    Standard joint multi-image RL.

    stack shape: (frames, y, x)
    psf: one common PSF or one PSF per frame
    """
    stack = np.asarray(stack, dtype=np.float64)
    stack = np.clip(stack, 0, None)

    if stack.ndim != 3:
        raise ValueError("stack must have shape (frames, y, x).")

    L = stack.shape[0]
    psfs = _prepare_psfs(psf, L)
    b = _prepare_backgrounds(backgrounds, stack)

    if init is None:
        mean_b = np.mean(b) if np.asarray(b).ndim == 1 else np.mean(b)
        x = np.full(stack.shape[1:], max(stack.mean() - mean_b, eps))
    else:
        x = np.clip(np.asarray(init, dtype=np.float64), eps, None)

    sensitivities = [
        adjoint_conv2(np.ones_like(x), p)
        for p in psfs
    ]

    for _ in range(iterations):
        corrections = []

        for l in range(L):
            bl = _background_for_frame(b, l)
            pred = conv2(x, psfs[l]) + bl
            ratio = stack[l] / np.maximum(pred, eps)

            c = adjoint_conv2(ratio, psfs[l])
            c /= np.maximum(sensitivities[l], eps)
            corrections.append(c)

        mean_correction = np.mean(corrections, axis=0)

        x *= mean_correction
        x = np.clip(x, 0, None)

    return x


def temporally_regularized_multi_image_rl(
    stack,
    psf,
    iterations=20,
    backgrounds=None,
    alpha=1.0,
    init=None,
    eps=EPS,
    return_history=False,
):
    """
    Temporally regularized multi-image Richardson-Lucy (TR-MIRL).

    At each iteration:
      1. Compute one standard RL correction map per frame.
      2. At each pixel, compute temporal mean and variance across correction maps.
      3. Convert temporal variance to a confidence weight:
             W = mu^2 / (mu^2 + alpha * var + eps)
      4. Apply a confidence-weighted update:
             update = 1 + W * (mu - 1)

    IMPORTANT:
    The regularization uses only the temporal axis. There is no radiality,
    gradient, local neighborhood correlation, or other spatial feature extraction.

    alpha:
      0   -> standard multi-image RL
      >0  -> increasing temporal regularization
    """
    stack = np.asarray(stack, dtype=np.float64)
    stack = np.clip(stack, 0, None)

    if stack.ndim != 3:
        raise ValueError("stack must have shape (frames, y, x).")

    L = stack.shape[0]
    if L < 2:
        raise ValueError("Temporal regularization requires at least 2 frames.")

    psfs = _prepare_psfs(psf, L)
    b = _prepare_backgrounds(backgrounds, stack)

    if init is None:
        mean_b = np.mean(b)
        x = np.full(stack.shape[1:], max(stack.mean() - mean_b, eps))
    else:
        x = np.clip(np.asarray(init, dtype=np.float64), eps, None)

    sensitivities = [
        adjoint_conv2(np.ones_like(x), p)
        for p in psfs
    ]

    history = {
        "mean_confidence": [],
        "mean_temporal_cv2": [],
        "relative_change": [],
    }

    for _ in range(iterations):
        corrections = np.empty_like(stack, dtype=np.float64)

        for l in range(L):
            bl = _background_for_frame(b, l)
            pred = conv2(x, psfs[l]) + bl
            ratio = stack[l] / np.maximum(pred, eps)

            c = adjoint_conv2(ratio, psfs[l])
            c /= np.maximum(sensitivities[l], eps)
            corrections[l] = c

        # PURELY TEMPORAL statistics: axis=0 is the frame/time axis.
        mu = np.mean(corrections, axis=0)
        var = np.var(corrections, axis=0, ddof=1)

        cv2 = var / np.maximum(mu**2, eps)
        W = 1.0 / (1.0 + alpha * cv2)

        update = 1.0 + W * (mu - 1.0)
        update = np.clip(update, eps, None)

        x_new = np.clip(x * update, 0, None)

        if return_history:
            rel = np.linalg.norm(x_new - x) / max(np.linalg.norm(x), eps)
            history["mean_confidence"].append(float(np.mean(W)))
            history["mean_temporal_cv2"].append(float(np.mean(cv2)))
            history["relative_change"].append(float(rel))

        x = x_new

    if return_history:
        return x, history
    return x



## 3. Optional alternative: pairwise temporal-consensus correction

This is closer to the idea of using pairwise temporal products.

For positive RL correction maps \(C_l\),

\[
P =
\frac{2}{L(L-1)}
\sum_{l<m} C_l C_m,
\qquad
C_{\mathrm{pair}} = \sqrt{P}.
\]

This is provided as an experimental alternative. The variance-weighted method above is the recommended starting point because it reduces to standard multi-image RL when \(\alpha=0\) and allows continuous control of the temporal regularization strength.


In [ ]:

def pairwise_temporal_consensus(corrections, eps=EPS):
    """
    Pairwise temporal product consensus.
    corrections shape: (frames, y, x)
    """
    C = np.asarray(corrections, dtype=np.float64)
    L = C.shape[0]

    if L < 2:
        raise ValueError("Need at least 2 frames.")

    # Efficient identity:
    # sum_{l<m} C_l C_m = 0.5 * [ (sum_l C_l)^2 - sum_l C_l^2 ]
    sum_c = np.sum(C, axis=0)
    sum_sq = np.sum(C**2, axis=0)

    pair_mean = (sum_c**2 - sum_sq) / (L * (L - 1))
    pair_mean = np.clip(pair_mean, 0, None)

    return np.sqrt(pair_mean + eps)


## 4. Synthetic test

In [ ]:

# Synthetic object
rng = np.random.default_rng(7)

ny, nx = 192, 192
truth = np.zeros((ny, nx), dtype=np.float64)

yy, xx = np.mgrid[:ny, :nx]
objects = [
    (55, 65, 3.0, 55),
    (62, 78, 3.0, 45),
    (120, 95, 5.0, 75),
    (110, 120, 2.5, 35),
    (80, 135, 4.0, 50),
]

for cy, cx, sigma_obj, amp in objects:
    truth += amp * np.exp(-((yy-cy)**2 + (xx-cx)**2) / (2*sigma_obj**2))

# Add a thin curved-like structure from several spots
for t in np.linspace(0, 1, 30):
    cy = 125 + 18*np.sin(2*np.pi*t)
    cx = 35 + 110*t
    truth += 13 * np.exp(-((yy-cy)**2 + (xx-cx)**2) / (2*1.4**2))

psf = gaussian_psf(size=21, sigma=2.2)

blurred = conv2(truth, psf)
background = 2.0

L = 20
synthetic_stack = np.stack([
    rng.poisson(np.clip(blurred + background, 0, None))
    for _ in range(L)
]).astype(np.float64)

print("Synthetic stack:", synthetic_stack.shape)
print("Mean photons/pixel/frame:", synthetic_stack.mean())


In [ ]:

iterations = 25

# 1) Single-frame RL
rec_single = single_image_rl(
    synthetic_stack[0],
    psf,
    iterations=iterations,
    background=background,
)

# 2) Sum of all frames -> RL -> divide by L to recover per-frame object scale
sum_image = synthetic_stack.sum(axis=0)
rec_sum = single_image_rl(
    sum_image,
    psf,
    iterations=iterations,
    background=L * background,
) / L

# 3) Standard multi-image RL
rec_multi = multi_image_rl(
    synthetic_stack,
    psf,
    iterations=iterations,
    backgrounds=background,
)

# 4) Temporally regularized multi-image RL
rec_temporal, history = temporally_regularized_multi_image_rl(
    synthetic_stack,
    psf,
    iterations=iterations,
    backgrounds=background,
    alpha=2.0,
    return_history=True,
)


In [ ]:

def mse(a, b):
    return np.mean((np.asarray(a) - np.asarray(b))**2)


def normalized_mse(reference, estimate):
    # Fit one global positive scale factor before computing MSE.
    # Useful for comparing reconstructions with slightly different intensity scale.
    r = np.asarray(reference, dtype=np.float64)
    e = np.asarray(estimate, dtype=np.float64)
    scale = np.sum(r * e) / max(np.sum(e * e), EPS)
    return mse(r, scale * e)


print(f"Normalized MSE | single frame RL : {normalized_mse(truth, rec_single):.5f}")
print(f"Normalized MSE | summed-stack RL : {normalized_mse(truth, rec_sum):.5f}")
print(f"Normalized MSE | multi-image RL  : {normalized_mse(truth, rec_multi):.5f}")
print(f"Normalized MSE | temporal RL     : {normalized_mse(truth, rec_temporal):.5f}")


In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

images = [
    (truth, "Ground truth"),
    (rec_single, "Single-frame RL"),
    (rec_sum, "Sum + RL"),
    (rec_multi, "Multi-image RL"),
    (rec_temporal, "Temporal RL"),
]

vmax = np.percentile(truth, 99.8)

for ax, (im, title) in zip(axes, images):
    ax.imshow(im, cmap="gray", vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(6, 4))
plt.plot(history["relative_change"])
plt.xlabel("Iteration")
plt.ylabel("Relative reconstruction change")
plt.title("TR-MIRL convergence")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history["mean_confidence"])
plt.xlabel("Iteration")
plt.ylabel("Mean temporal confidence")
plt.title("Mean temporal confidence")
plt.show()



## 5. Apply the method to your own TIFF stack

### Option A — one multi-page TIFF

Set:

```python
STACK_PATH = "your_stack.tif"
```

### Option B — a folder containing one TIFF per frame

Use:

```python
stack, files = load_tiff_folder("folder_with_frames")
```

Before running the temporal method, the frames should ideally be:

- registered to the same spatial coordinates,
- acquired from a static or slowly changing specimen,
- corrected for obvious detector offsets,
- checked for photobleaching or large illumination fluctuations.

The temporal regularizer assumes that frame-to-frame inconsistency is predominantly noise. Real sample motion or real biological dynamics will also appear as temporal inconsistency.


In [ ]:

# ---------- USER SETTINGS ----------

STACK_PATH = "your_stack.tif"

# If you have a measured PSF, set USE_MEASURED_PSF = True
USE_MEASURED_PSF = False
PSF_PATH = "measured_psf.tif"

# Otherwise generate a Gaussian PSF.
# sigma is in pixels.
GAUSSIAN_PSF_SIZE = 21
GAUSSIAN_PSF_SIGMA = 2.0

ITERATIONS = 20

# Temporal regularization strength.
# alpha = 0 -> standard multi-image RL
# Suggested initial sweep: 0, 0.25, 0.5, 1, 2, 4
ALPHA = 1.0

# Background:
# "zero"       : use if data are already background/dark corrected
# "percentile" : estimate one scalar background per frame
BACKGROUND_MODE = "percentile"
BACKGROUND_PERCENTILE = 5


In [ ]:

# Load data
stack = load_tiff_stack(STACK_PATH)

print("Stack shape:", stack.shape)
print("Frames:", stack.shape[0])
print("Image size:", stack.shape[1:])
print("Min / max:", stack.min(), stack.max())
print("Mean photons/intensity per pixel:", stack.mean())

# PSF
if USE_MEASURED_PSF:
    psf_user = tiff.imread(PSF_PATH).astype(np.float64)
    if psf_user.ndim != 2:
        raise ValueError("Measured PSF must be a 2D TIFF.")
    psf_user = normalize_psf(psf_user)
else:
    psf_user = gaussian_psf(
        size=GAUSSIAN_PSF_SIZE,
        sigma=GAUSSIAN_PSF_SIGMA,
    )

# Background
if BACKGROUND_MODE == "zero":
    backgrounds_user = np.zeros(stack.shape[0])
elif BACKGROUND_MODE == "percentile":
    backgrounds_user = estimate_scalar_backgrounds(
        stack,
        percentile=BACKGROUND_PERCENTILE,
    )
else:
    raise ValueError("Unknown BACKGROUND_MODE.")

print("Estimated backgrounds:", backgrounds_user)


In [ ]:

# Visualize a few raw frames and the temporal mean

n_show = min(4, stack.shape[0])

fig, axes = plt.subplots(1, n_show + 1, figsize=(4*(n_show+1), 4))

for i in range(n_show):
    axes[i].imshow(stack[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")

axes[-1].imshow(stack.mean(axis=0), cmap="gray")
axes[-1].set_title("Temporal mean")
axes[-1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# Run the three main comparisons

# A. Sum + RL
sum_image_user = stack.sum(axis=0)
rec_sum_user = single_image_rl(
    sum_image_user,
    psf_user,
    iterations=ITERATIONS,
    background=np.sum(backgrounds_user),
) / stack.shape[0]

# B. Standard multi-image RL
rec_multi_user = multi_image_rl(
    stack,
    psf_user,
    iterations=ITERATIONS,
    backgrounds=backgrounds_user,
)

# C. Temporally regularized multi-image RL
rec_temporal_user, history_user = temporally_regularized_multi_image_rl(
    stack,
    psf_user,
    iterations=ITERATIONS,
    backgrounds=backgrounds_user,
    alpha=ALPHA,
    return_history=True,
)


In [ ]:

# Compare results

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

display_images = [
    (stack.mean(axis=0), "Raw temporal mean"),
    (rec_sum_user, "Sum + RL"),
    (rec_multi_user, "Multi-image RL"),
    (rec_temporal_user, f"Temporal RL (alpha={ALPHA})"),
]

# Common display range for easier visual comparison
all_vals = np.concatenate([im.ravel() for im, _ in display_images])
vmin = np.percentile(all_vals, 1)
vmax = np.percentile(all_vals, 99.8)

for ax, (im, title) in zip(axes, display_images):
    ax.imshow(im, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()



## 6. Sweep the temporal regularization strength

The parameter \(\alpha\) controls how strongly temporal inconsistency suppresses RL updates.

- \(\alpha = 0\): standard multi-image RL
- small \(\alpha\): mild temporal denoising
- large \(\alpha\): stronger suppression of temporally inconsistent corrections

Too large an \(\alpha\) can suppress weak but real features, so it should be validated using simulations, beads, or another known sample.


In [ ]:

alpha_values = [0, 0.25, 0.5, 1, 2, 4]

alpha_results = {}

for a in alpha_values:
    alpha_results[a] = temporally_regularized_multi_image_rl(
        stack,
        psf_user,
        iterations=ITERATIONS,
        backgrounds=backgrounds_user,
        alpha=a,
    )

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, a in zip(axes.ravel(), alpha_values):
    ax.imshow(alpha_results[a], cmap="gray")
    ax.set_title(f"alpha = {a}")
    ax.axis("off")

plt.tight_layout()
plt.show()



## 7. Inspect the temporal confidence map

The confidence map is not a spatial-feature map. It is computed independently at every pixel from the temporal variability of the frame-specific RL corrections.

A value close to 1 means the correction is highly consistent across frames.

A lower value means the proposed RL correction varies substantially from frame to frame.


In [ ]:

def temporal_confidence_map(stack, x, psf, backgrounds=None, alpha=1.0, eps=EPS):
    stack = np.asarray(stack, dtype=np.float64)
    L = stack.shape[0]
    psfs = _prepare_psfs(psf, L)
    b = _prepare_backgrounds(backgrounds, stack)

    corrections = np.empty_like(stack, dtype=np.float64)

    for l in range(L):
        sensitivity = adjoint_conv2(np.ones_like(x), psfs[l])
        bl = _background_for_frame(b, l)

        pred = conv2(x, psfs[l]) + bl
        ratio = stack[l] / np.maximum(pred, eps)

        c = adjoint_conv2(ratio, psfs[l])
        c /= np.maximum(sensitivity, eps)
        corrections[l] = c

    mu = corrections.mean(axis=0)
    var = corrections.var(axis=0, ddof=1)
    cv2 = var / np.maximum(mu**2, eps)
    W = 1.0 / (1.0 + alpha * cv2)

    return W, mu, var


W_user, correction_mean_user, correction_var_user = temporal_confidence_map(
    stack,
    rec_temporal_user,
    psf_user,
    backgrounds=backgrounds_user,
    alpha=ALPHA,
)

plt.figure(figsize=(6, 5))
plt.imshow(W_user, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Temporal confidence")
plt.title("Pixelwise temporal confidence")
plt.axis("off")
plt.show()


## 8. Save the reconstructions

In [ ]:

output_dir = Path("TR_MIRL_output")
output_dir.mkdir(exist_ok=True)

tiff.imwrite(output_dir / "raw_temporal_mean.tif", stack.mean(axis=0).astype(np.float32))
tiff.imwrite(output_dir / "sum_plus_RL.tif", rec_sum_user.astype(np.float32))
tiff.imwrite(output_dir / "standard_multiimage_RL.tif", rec_multi_user.astype(np.float32))
tiff.imwrite(output_dir / "temporally_regularized_multiimage_RL.tif", rec_temporal_user.astype(np.float32))
tiff.imwrite(output_dir / "temporal_confidence_map.tif", W_user.astype(np.float32))

print("Saved results to:", output_dir.resolve())



## 9. Important interpretation

This implementation should be considered an **experimental temporally regularized RL method**, not a standard established RL variant.

For repeated images with the same PSF and purely independent Poisson noise, standard joint multi-image RL and RL applied to the summed photon image contain essentially the same likelihood information.

The purpose of the temporal regularizer here is different: it introduces a prior that **RL corrections that are reproducible across repeated acquisitions are more trustworthy than corrections that fluctuate strongly from frame to frame**.

A rigorous validation should therefore compare, at equal total photon counts:

1. single-frame RL,
2. summed-stack RL,
3. standard multi-image RL,
4. temporally regularized multi-image RL.

Useful quantitative metrics include reconstruction error in simulations, bead FWHM/FRC for resolution, background standard deviation, SNR/CNR, and preservation of integrated fluorescence intensity.
